# Stage 3: CyTOF annotation

This stage loads the GFP-high z-scored dataframe produced by preprocessing, runs UMAP and Leiden diagnostics, assigns manual UMAP regions, fills remaining labels with the source XGBoost configuration, and exports the labeled dataframe.


In [ ]:
# ruff: noqa: E402, E501
from pathlib import Path
import sys

# Support launching Jupyter from either the checkout root or its notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
# Prefer this checkout over another installed copy or worktree of app.
sys.path.insert(0, str(PROJECT_ROOT))
import app  # noqa: E402

print(f"Kernel executable: {sys.executable}")
print(f"Working directory: {Path.cwd().resolve()}")
print(f"Imported app package: {Path(app.__file__).resolve()}")

# Fail early if Python resolved app from outside this checkout.
if Path(app.__file__).resolve().parent != PROJECT_ROOT / "app":
    raise RuntimeError("app was imported from outside the intended checkout")

In [ ]:
from app.config import ANNOTATION_FILE, OUTPUT_ROOT, SKIP_BATCH_CORRECTION, SUBSAMPLE, UMAP_DIR
from app.utils import attach_annotation_labels
import os
import pandas as pd
from IPython.display import display
from app.annotation import (
    CELL_IDENTITY_MARKERS,
    BASAL_CLASSICAL_SEPARATING_MARKERS,
    calculate_umap,
    calc_leiden,
    plot_cluster_heatmap,
    plot_condition_umap,
    plot_full_umap,
    plot_clusters,
    plot_clusters_by_condition,
    plot_umap_class_probabilities,
    plot_umap_staining,
    predict_noise_xgboost,
    save_labeled_data,
)

PLOT_ALL_UMAP_STAINING = os.getenv("PLOT_ALL_UMAP_STAINING")

Run paths

## Load Data

In [ ]:
BALANCED_GFP_HIGH_FILE = "GFP_high_ZSCORED_BATCH_CORRECTED_BALANCED_TREATMENT.csv"
FULL_GFP_HIGH_FILE = "all_GFP_high_ZSCORED_BATCH_CORRECTED.csv"
selected = BALANCED_GFP_HIGH_FILE


pooled_scaled_FILE = OUTPUT_ROOT / "preprocess" / selected
print(pooled_scaled_FILE)
if not (pooled_scaled_FILE).is_file():
    raise FileNotFoundError(pooled_scaled_FILE)

pooled_scaled = pd.read_csv(pooled_scaled_FILE, low_memory=False)
pooled_scaled = attach_annotation_labels(pooled_scaled, str(ANNOTATION_FILE))
print(f"pooled_scaled shape:", pooled_scaled.shape)
pooled_scaled.columns

## 1. Dimensionality reduction

Fit UMAP from the selected cell-identity markers once, then reuse the Parquet cache on later runs.


In [ ]:
UMAP_ROOT = OUTPUT_ROOT / "6. plots - GFP+ analyses" / "GFP_high_ZSCORED_UMAP_by_marker_group"
UMAP_CACHE_FILE = UMAP_DIR / f"{selected}.parquet"
df_umap = calculate_umap(
    pooled_scaled.copy(),
    CELL_IDENTITY_MARKERS,
    UMAP_CACHE_FILE,
)

#### Review the embedding

In [ ]:
plot_full_umap(df_umap.copy(), UMAP_ROOT / "cell_identity_all_cells.png")
plot_condition_umap(df_umap.copy(), UMAP_ROOT / "cell_identity_by_condition.png")

## 3. Manual regions and biological labels

Seed classical and basal regions with the original UMAP inequalities. Fit XGBoost on the aligned scaled GAL4, GATA6, and TSPAN8 values to fill only unassigned rows.

In [ ]:
label_dir = OUTPUT_ROOT / "7. Clustering" / "Label"

plot_umap_staining(
    df_umap.copy(), label_dir, marker_group="cell_identity", markers=("GAL4", "GATA6", "TSPAN8")
)

# if (
#     not pooled_scaled["cell_uid"]
#     .reset_index(drop=True)
#     .equals(df_umap["cell_uid"].reset_index(drop=True))
# ):
#     raise ValueError("Pooled scaled expression and UMAP coordinates are not cell-UID/order aligned")

#### Plot the labeling results

In [ ]:
import numpy as np

data = df_umap.copy()
manual_labels = np.full(len(data), -1, dtype=int)


# subsample
# manual_labels[
#     (data["UMAP1"].to_numpy() > 6) & (data["UMAP2"].to_numpy() < 6) & (data["UMAP2"].to_numpy() > 1)
# ] = 0

manual_labels[(data["UMAP1"].to_numpy() < 0) & (data["UMAP2"].to_numpy() > 6)] = 1

manual_labels[(data["UMAP1"].to_numpy() > 8)] = 1
manual_labels[(data["UMAP1"].to_numpy() > 8) & (data["UMAP2"].to_numpy() < 11)] = 0

manual_labels[
    (data["UMAP1"].to_numpy() > 4) & (data["UMAP2"].to_numpy() > 6) & (data["UMAP2"].to_numpy() < 9)
] = 0


manual_labels = pd.Series(manual_labels).replace({0: "classic", 1: "basal"})

plot_clusters(
    df_umap.copy(), manual_labels, "Manual labeling regions", label_dir / "manual_clusters.png"
)


manual_labels = pd.Series(manual_labels).replace({"classic": 0, "basal": 1})
predicted_labels, predicted_probabilities, xgb_model = predict_noise_xgboost(
    pooled_scaled.copy(),
    manual_labels,
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    n_jobs=1,
    features=("GAL4", "GATA6", "TSPAN8"),
)
predicted_labels = pd.Series(predicted_labels).replace({0: "classic", 1: "basal"})


plot_clusters(
    df_umap.copy(),
    predicted_labels,
    "XGBoost | All cells assigned",
    label_dir / "XGBoost_clusters.png",
)
if predicted_probabilities is not None and xgb_model is not None:
    plot_umap_class_probabilities(
        df_umap.copy(),
        predicted_probabilities,
        xgb_model.classes_,
        label_dir / "XGBoost_probabilities_by_class.png",
    )

#### Export the labeled data

## 2. Leiden clustering diagnostics

Use the cached coordinates with the source symmetric kNN graph, RBConfiguration partition, 100 neighbors, resolution 0.07, seed 42, and two Leiden iterations.


In [ ]:
leiden_labels = calc_leiden(df_umap.copy(), n_neighbors=100, resolution=0.07, random_state=42)
leiden_dir = OUTPUT_ROOT / "7. Clustering" / "GFP_high_ZSCORED_UMAP_Leiden"
leiden_name = "all_cell_identity_Leiden_neighbors100_resolution0.07"
plot_clusters(
    df_umap.copy(),
    leiden_labels,
    "Leiden | all | cell_identity",
    leiden_dir / f"{leiden_name}.png",
    point_size=4,
    figsize=(9, 7),
)
plot_clusters_by_condition(
    df_umap.copy(),
    leiden_labels,
    ["untreated", "treated"],
    leiden_dir / "Leiden_clusters_by_condition.png",
    figsize=(9, 14),
    point_size=4,
    alpha=0.8,
)
plot_cluster_heatmap(
    df_umap.copy(),
    leiden_labels,
    CELL_IDENTITY_MARKERS,
    leiden_dir / "heatmap_cell_identity.png",
    title="Cell identity markers by Leiden cluster",
    figsize=(23, 23),
)
plot_cluster_heatmap(
    df_umap.copy(),
    leiden_labels,
    BASAL_CLASSICAL_SEPARATING_MARKERS,
    leiden_dir / "heatmap_selected_features.png",
    title="Selected markers by Leiden cluster",
    figsize=(23, 23),
)

## 4. Create Annotations

In [ ]:
annotation_map = pd.DataFrame(index=pooled_scaled.index)
annotation_map["cell_uid"] = pooled_scaled["cell_uid"]
annotation_map["labels"] = predicted_labels
annotation_map["leiden_labels"] = leiden_labels.astype(int)


annotation_map.to_csv(ANNOTATION_FILE, index=False)
print(f"Saved cell annotation map: {ANNOTATION_FILE}")
display(annotation_map["labels"].value_counts(dropna=False))

## 5. Staining

In [ ]:
if PLOT_ALL_UMAP_STAINING:
    staining_summary = pd.DataFrame(
        plot_umap_staining(
            df_umap.copy(),
            OUTPUT_ROOT / "6. plots - GFP+ analyses" / "GFP_high_ZSCORED_UMAP_marker_staining",
            marker_group="cell_identity",
        )
    )
    staining_summary.to_csv(
        OUTPUT_ROOT / "6. plots - GFP+ analyses" / "UMAP_marker_staining_summary.csv", index=False
    )
    display(staining_summary)